In [0]:
%pip install openai

In [0]:
%pip install ruamel.yaml

In [0]:
dbutils.library.restartPython()

In [0]:
#testing the connection to claude

from openai import OpenAI

# Workspace URL
workspace_url = "https://" + spark.conf.get("spark.databricks.workspaceUrl")

# Databricks authentication
token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

# Connect to Claude through Databricks
client = OpenAI(
    api_key=token,
    base_url=f"{workspace_url}/ai-gateway/mlflow/v1"
)

# Test Claude
response = client.chat.completions.create(
    model="system.ai.claude-opus-4-7",
    messages=[
        {
            "role": "user",
            "content": "Explain what persist_docs does in dbt in two sentences."
        }
    ],
    max_tokens=500
)

print(response.choices[0].message.content)

In [0]:
# Project configuration
# No user-specific paths or environment names are hardcoded:
# - PROJECT_PATH is derived from this notebook's location (<repo>/notebooks/<notebook>)
# - CATALOG and SCHEMA are provided via notebook widgets

import os

notebook_path = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

PROJECT_PATH = "/Workspace" + os.path.dirname(os.path.dirname(notebook_path))

DBT_PROJECT_PATH = f"{PROJECT_PATH}/dbt_project"

SCHEMA_YAML_PATH = f"{DBT_PROJECT_PATH}/models/gold/schema.yml"

dbutils.widgets.text("catalog", "", "Unity Catalog catalog")
dbutils.widgets.text("schema", "", "Unity Catalog schema")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()

if not CATALOG or not SCHEMA:
    raise ValueError("Set the 'catalog' and 'schema' widgets before running this notebook.")

TABLES = [
    "gold_dim_customer",
    "gold_dim_date",
    "gold_dim_energy_tariff",
    "gold_fact_energy_consumption"
]

print("Project:", PROJECT_PATH)
print("YAML:", SCHEMA_YAML_PATH)

In [0]:
# checking if we can read the yaml
import os

print("Project exists:", os.path.exists(PROJECT_PATH))
print("dbt project exists:", os.path.exists(DBT_PROJECT_PATH))
print("Schema YAML exists:", os.path.exists(SCHEMA_YAML_PATH))



In [0]:
#extracting metadata

tables_metadata = {}

for table in TABLES:
    full_table_name = f"{CATALOG}.{SCHEMA}.{table}"

    df = spark.table(full_table_name)

    columns = [
        {
            "name": field.name,
            "type": field.dataType.simpleString()
        }
        for field in df.schema.fields
    ]

    sample = [
        row.asDict() for row in df.limit(5).collect()
    ]

    tables_metadata[table] = {
        "columns": columns,
        "sample_data": sample
    }

print("Metadata extracted successfully!")

for table, metadata in tables_metadata.items():
    print(table, "-", len(metadata["columns"]), "columns")

In [0]:
#preparing the prompt for claude 
import json

SYSTEM_PROMPT = """
You are a senior data architect specializing in energy analytics.

You are documenting the GOLD layer of Alpenstrom,
a fictional Swiss energy provider.

Business context:
- Country: Switzerland
- Currency: CHF
- Energy consumption: MWh
- Peak demand: kW
- Carbon emissions: metric tons
- Customers operate across Swiss cantons and language regions.

Your task is to generate descriptions for dbt tables and columns.

Requirements:
1. Write in clear, professional English.
2. Explain business meaning, not just technical names.
3. Make descriptions useful for conversational analytics with Databricks Genie.
4. Use the provided sample data to understand actual values.
5. Never invent business rules, primary keys or foreign keys.
6. Never assume accounting policies or calculation formulas.
7. If information is insufficient, use neutral language.
8. Return valid JSON only.

Expected JSON structure:
{
  "table_description": "...",
  "columns": [
    {
      "name": "column_name",
      "description": "..."
    }
  ]
}
"""

In [0]:
# creating the generate documentation function 
def generate_documentation(table_name, metadata):

    user_prompt = f"""
    Generate documentation for this dbt model.

    Table name:
    {table_name}

    Columns and data types:
    {json.dumps(metadata["columns"], indent=2)}

    Sample records:
    {json.dumps(metadata["sample_data"], indent=2, default=str)}
    """

    response = client.chat.completions.create(
        model="system.ai.claude-opus-4-7",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=4000
    )

    return json.loads(response.choices[0].message.content)

In [0]:
from ruamel.yaml import YAML
import shutil
from datetime import datetime

# 1. Generate documentation for all tables
all_documentation = {}

for table_name, metadata in tables_metadata.items():

    print(f"Generating documentation: {table_name}")

    all_documentation[table_name] = generate_documentation(
        table_name,
        metadata
    )

# 2. Create a backup of the existing YAML
backup_path = SCHEMA_YAML_PATH + ".backup"

shutil.copy2(SCHEMA_YAML_PATH, backup_path)

# 3. Load existing schema.yml
yaml = YAML()
yaml.preserve_quotes = True

with open(SCHEMA_YAML_PATH, "r") as file:
    schema_data = yaml.load(file)

# 4. Update model and column descriptions
for model in schema_data["models"]:

    table_name = model["name"]

    if table_name not in all_documentation:
        continue

    documentation = all_documentation[table_name]

    model["description"] = documentation["table_description"]

    generated_columns = {
        col["name"]: col["description"]
        for col in documentation["columns"]
    }

    for column in model["columns"]:
        column_name = column["name"]

        if column_name in generated_columns:
            column["description"] = generated_columns[column_name]

# 5. Save updated YAML
with open(SCHEMA_YAML_PATH, "w") as file:
    yaml.dump(schema_data, file)

print("Documentation successfully generated!")
print("schema.yml updated!")
print("Backup created:", backup_path)

In [0]:
# checking if the documentation was effectively written

with open(SCHEMA_YAML_PATH, "r") as file:
    print(file.read())